# Hemoflow Verification Document

This document outlines the process for verifying the HemoFlow C++ software package, which is based on the Palabos lattice–Boltzmann CFD solver framework. As the Palabos package is already well verified, our test cases focus only on the domain-specific features, such as the arbitrary parabolic inlet profile, Murray’s law-based outlet flow rates, and porous-layer stent modelling.

Three test cases were prepared:

- Straight pipe flow (Poiseuille flow) using our custom voxelisation process
- Realistic sidewall aneurysm from the Aneurisk database (https://ecm2.mathcs.emory.edu/aneuriskweb/about)
- Realistic sidewall aneurysm from the AneuX morphology database (https://github.com/hirsch-lab/aneuxdb)

Importing python packages for evaluation

In [ ]:
import pandas as pd
import seaborn as sns

## Straight pipe flow (Poiseuille flow) 

As cerebral vessel sections can be represented as a branched system of curved pipes, our initial verification was performed on a straight pipe flow, commonly referred to as Poiseuille flow ([https://en.wikipedia.org/wiki/Hagen–Poiseuille_equation](https://en.wikipedia.org/wiki/Hagen%E2%80%93Poiseuille_equation)). This case provides an analytical solution for both the fully developed velocity profile and the pressure drop across a given section.

The following features were examined, and their outcomes are summarised below:

- Startup ramping process to eliminate transient effects from the initial condition
    - For a straight geometry, a pressure wave was observed, which required a long time to decay. When tested on a patient-specific case, these waves dissipated almost immediately due to the pronounced curvature of the vessel section.
- Time-dependent volume flow rate (VFR) condition read from a text file
    - The normalised VFR curve values are read from text files, and the parabolic velocity profile is scaled according to the prescribed boundary condition velocity magnitude and the normalised curve.
- Parabolic velocity boundary condition setup
    - The algorithm can impose velocity profiles in arbitrary directions and positions, which is crucial since outlets may lie near the corners of the computational domain. The boundary condition can also prescribe both inlet and outlet VFR values.
- Zero-pressure boundary condition
    - As noted previously, this condition does not dampen pressure waves.
- Porous layer resistance applied to stent voxels
    - Since this feature is central to our current application, we examined its dependency on voxel size as well as the influence of stent surface curvature.

### Voxel size dependence

#### Pressure drop

In the straight pipe scenario, we observed that the pressure wave oscillated between the inlet and the outlet while using a constant pipe flow profile at the inlet. To address this oscillation, a ramping inlet condition was applied, increasing the flow rate from zero to the desired volume flow rate at $t=0$ over the first half of the warm-up phase duration. The duration of the warm-up phase iterations was based on the maximum number of voxels in the x, y, or z direction.

The sensitivity analysis of the pressure drop indicates a notable difference between the 0.3 mm and 0.2 mm cases, while the convergence for smaller dx values remains less consistent. This behaviour may be attributed to the presence of residual pressure waves during the warm-up phase. However, due to computational limitations, the warm-up phase was not extended. We expect that in highly curved blood vessel geometries, these artificial pressure waves will dissipate quickly.

In [ ]:
pipe_dx=pd.read_csv('./input/pipe_dx_dp_sensitivity.csv')
pipe_dx.drop(index=pipe_dx.index[0], inplace=True)
pipe_dx.drop(columns=["Unnamed: 0", 'iteration'], inplace=True)
pipe_dx

In [ ]:
ax= sns.scatterplot(pipe_dx, x='dx', y='pressure_drop')
ax.set(title='Poiseuille flow: Pressure drop in function of voxel size')

#### Pressure drop with a straight flow diverter

TODO

#### Pressure drop with a curved flow diverter

TODO

#### Velocity profiles

Velocity profiles were extracted just before the outlet boundary condition along the x-axis. The voxel resolution used was 0.3 mm for the coarse grid and 0.08 mm for the fine grid. The coarse profile demonstrates good agreement with the analytical profile, with only the interpolation error from the post-processing algorithm being noticeable. In the fine grid, a minor asymmetry is visible, which remains under investigation.

##### Coarse

![](input/profile_comp_coarse.png "coarse profile")

##### Fine

![](input/profile_comp_fine.png "fine profile")

TODO: asymmetry!

### Womersley profile
TODO

## Aneurisk testcase

Our primary test case involves a sidewall aneurysm model with a nominal vessel diameter of 4 mm. We deployed a 4 mm nominal Pipeline Embolization Device (PED) type stent. The inlet flow rate follows a population-averaged internal carotid artery (ICA) waveform at 72 beats per minute. We normalised the amplitude of the waveform. The average inlet velocity is set to 0.7 m/s, which represents the maximum value reported in the literature.

The main output we analysed is the space- and time-averaged velocity within the aneurysm sac. We evaluated this using our in-house post-processing code (LBMpost), which has not yet been open-sourced. We also conducted sensitivity analyses for spatial resolution (dx), time step (dt), and save frequency.

### Voxel size dependence

We assessed the dependence on voxel size using six different resolutions. The results demonstrated good convergence. We selected an optimal voxel size of 0.08 mm, where the difference from the highest resolution was only 3%.

In [ ]:
aneurisk_dx=pd.read_csv('./input/aneurisk_dx_STAV_sensitivity.csv')
aneurisk_dx.drop(index=aneurisk_dx.index[0], inplace=True)
aneurisk_dx.drop(columns=["Unnamed: 0", 'iteration'], inplace=True)
aneurisk_dx

In [ ]:
ax=sns.scatterplot(aneurisk_dx, x='dx', y='ane_0_velocity_vol_avg')
ax.set(title='Aneurysm flow: Space and time averaged velocity in function of voxel size')

### Timestep length dependence

We evaluated the dependence on time step length at a fixed voxel resolution of 0.08 mm. Although the convergence was less pronounced than for spatial resolution, the effect of the time step length was significantly smaller. However, larger time steps occasionally led to divergence, particularly in regions with high velocities due to geometric features. For stability, we chose a time step length of 1e-5 seconds.

In [ ]:
aneurisk_dt=pd.read_csv('./input/aneurisk_dt_STAV_sensitivity.csv')
aneurisk_dt.drop(index=aneurisk_dt.index[0], inplace=True)
aneurisk_dt.drop(columns=["Unnamed: 0", 'iteration'], inplace=True)
aneurisk_dt

In [ ]:
ax=sns.scatterplot(aneurisk_dt, x='dt', y='ane_0_velocity_vol_avg')
ax.set(title='Aneurysm flow: Space and time averaged velocity in function of timestep size')

### Save time dependency

We also examined the effect of save frequency over a 0.8 second cardiac cycle. As expected, when using a robust parameter set, the sampling frequency had a minimal impact on the averaged values. However, we compared transient signals and found that a sampling interval of 0.016 seconds was sufficient. This corresponds to saving and evaluating 50 time steps.

In [ ]:
aneurisk_save_dt=pd.read_csv('./input/aneurisk_savedt_STAV_sensitivity.csv')
aneurisk_save_dt.drop(index=aneurisk_save_dt.index[0], inplace=True)
aneurisk_save_dt.drop(columns=["Unnamed: 0", 'iteration'], inplace=True)
aneurisk_save_dt

In [ ]:
ax=sns.scatterplot(aneurisk_save_dt, x='save_dt', y='ane_0_velocity_vol_avg')
ax.set(title='Aneurysm flow: Space and time averaged velocity in function of save frequency')